# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
!pip install kagglehub

import os
import shutil
import kagglehub

# 1. Download the latest version of the aviation accident dataset from Kaggle
downloaded_path = kagglehub.dataset_download("mos3santos/acidentes-de-aviao-at-2023")
print("Downloaded to cache folder:", downloaded_path)

# 2. Automatically find the CSV file and copy it directly into your active folder
for file in os.listdir(downloaded_path):
    if file.endswith('.csv'):
        # Copy file to your current active folder directory
        shutil.copy(os.path.join(downloaded_path, file), os.getcwd())
        print(f"✅ Successfully copied '{file}' into your active project folder!")


Downloaded to cache folder: /home/koinange/.cache/kagglehub/datasets/mos3santos/acidentes-de-aviao-at-2023/versions/1
✅ Successfully copied 'USState_Codes.csv' into your active project folder!
✅ Successfully copied 'AviationData.csv' into your active project folder!


In [3]:
aviation_df=pd.read_csv('AviationData.csv',encoding="iso-8859-1")
print(aviation_df.shape)

Sprint(aviation_df.info)

print(aviation_df.isna().sum())

print(aviation_df.describe)

(88889, 31)


/home/koinange/anaconda3/envs/learn-env/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3145: DtypeWarning: Columns (6,7,28) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


NameError: name 'Sprint' is not defined

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
aviation_df['Event.Date'] = pd.to_datetime(aviation_df['Event.Date'], errors='coerce')
aviation_df['Year'] = aviation_df['Event.Date'].dt.year
aviation_df = aviation_df[(aviation_df['Year'] >= 1983) & (aviation_df['Year'] <= 2023)]
aviation_df['Aircraft.Category'] = aviation_df['Aircraft.Category'].fillna('')
aviation_df = aviation_df[aviation_df['Aircraft.Category'].str.lower() == 'airplane']
print(aviation_df.shape)

(24441, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
# CLEANING ASSUMPTION: 
# Missing values (NaNs) in injury/survival columns imply that zero occupants 
# sustained that specific category of harm. We impute these to 0.0 to prevent 
# mathematical failures and data dropouts during row additions.

injury_columns = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
for col in injury_columns:
    aviation_df[col] = aviation_df[col].fillna(0)
    
    
# DERIVED COLUMN: 'Total.Onboard' to evaluate casualty rates as a standardized ratio, we must first 
# establish the total human footprint baseline on each aircraft. We sum all 
# injured and uninjured categories row-by-row to reconstruct total occupancy.

aviation_df['Total.Onboard'] = (
    aviation_df['Total.Fatal.Injuries'] + 
    aviation_df['Total.Serious.Injuries'] + 
    aviation_df['Total.Minor.Injuries'] + 
    aviation_df['Total.Uninjured']
)


# DERIVED COLUMN: 'Injury.Rate' that quantifies the probability of severe passenger trauma. It isolates 
# combined fatal and serious injuries as a fractional percentage of total onboard 
# occupants. We use np.where as a guardrail to assign a safe 0.0 baseline to 
# empty, parked aircraft incidents, preventing zero-division errors.

aviation_df['Injury.Rate'] = np.where(
    aviation_df['Total.Onboard'] > 0,
    (aviation_df['Total.Fatal.Injuries'] + aviation_df['Total.Serious.Injuries']) / aviation_df['Total.Onboard'],
    0.0
)


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# We fill missing fields with 'Unknown', strip out loose spacing, and force a 
# uniform title-case formatting structure so strings match perfectly.
aviation_df['Aircraft.damage'] = aviation_df['Aircraft.damage'].fillna('Unknown').str.strip().str.title()

# DERIVED COLUMN: 'Is.Destroyed' to help our insurer client isolate aircraft models with high rates 
# of hull loss, we transform the descriptive categories into a binary metric. 
# Catastrophic structural write-offs ('Destroyed') are flagged as 1, while 
# manageable incidents (Minor/Substantial) are flagged as 0.

aviation_df['Is.Destroyed'] = np.where(aviation_df['Aircraft.damage'] == 'Destroyed', 1, 0)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

Over a 40-year dataset span, manufacturing brand names are often entered with inconsistent casings, trailing spaces, and regional corporate suffixes. 
We force all entries to uppercase and strip outer spaces to eliminate variants.

We use an explicit translation dictionary mapping to collapse fragmented corporate name variations (like 'BOEING COMPANY') into unified core organizational labels (like 'BOEING'). This prevents the machine from treating them as separate entities. 

Per the client's strict mandate for high statistical stability, we evaluate total brand frequencies and filter out rare or experimental manufacturers with fewer than 50 recorded incidents. This eliminates volatile outlier anomalies.

In [ ]:
aviation_df['Make'] = aviation_df['Make'].fillna('UNKNOWN').str.strip()
make_clean_dict = {
    'CESSNA AIRCRAFT CO': 'CESSNA', 'CESSNA AIRCRAFT': 'CESSNA',
    'PIPER AIRCRAFT INC': 'PIPER', 'PIPER AIRCRAFT': 'PIPER',
    'BEECH AIRCRAFT': 'BEECH', 'BEECH AIRCRAFT CORP': 'BEECH', 'BEECHCRAFT': 'BEECH',
    'BOEING COMPANY': 'BOEING'
}
aviation_df['Make'] = aviation_df['Make'].replace(make_clean_dict)

make_sample_frequencies = aviation_df['Make'].value_counts()
robust_make_indices = make_sample_frequencies[make_sample_frequencies >= 50].index
aviation_df = aviation_df[aviation_df['Make'].isin(robust_make_indices)]


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

IMPUTATION METHODOLOGY: Statistical Mode Replacement
Engine counts must be realistic integers—imputing missing values with a statistical mean (average) would yield impossible decimals like 1.7 engines.
We force the column to numeric formatting and impute missing records using the column's Mode (the most frequently occurring value). This maintains absolute physical and structural accuracy across our insurance risk metrics.

In [ ]:
categorical_features = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for column_name in categorical_features:
    aviation_df[column_name] = aviation_df[column_name].fillna('Unknown').str.strip().str.title()
    
engine_mode_series = aviation_df['Number.of.Engines'].mode()
default_engine_mode = engine_mode_series.iloc[0] if not engine_mode_series.empty else 1.0

aviation_df['Number.of.Engines'] = aviation_df['Number.of.Engines'].fillna(default_engine_mode)    

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
categorical_features = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']
for column_name in categorical_features:
    aviation_df[column_name] = aviation_df[column_name].fillna('Unknown').str.strip().str.title()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
columns_to_prune = ['Airport.Code', 'Airport.Name', 'Latitude', 'Longitude']
aviation_df = aviation_df.drop(columns=[col for col in columns_to_prune if col in aviation_df.columns])

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
aviation_df.to_csv("AviationData_Cleaned_Master.csv", index=False)
print(aviation_df.shape)

(17469, 31)
